# Notebook Overview — Run Qwen2-VL Baseline

## Purpose

This notebook performs development-subset baseline Video Question Answering (VideoQA) experiments for the NExT-QA benchmark dataset using the Qwen2-VL-7B multimodal foundation model. The workflow establishes a baseline performance reference against which all representation-based VideoQA pipelines (CLIP-based and self-supervised autoencoder approaches) are evaluated.

Development-subset experiments are used to enable rapid iteration, debugging, and parameter validation while minimizing computational cost. The resulting baseline provides a controlled reference point for comparing downstream representation learning approaches before scaling to full-dataset evaluation.

All outputs are stored as experiment artifacts under a structured Google Drive experiment directory to ensure reproducibility and cross-experiment comparison.

---

## Inputs

* NExT-QA video dataset (Google Drive source)
* NExT-QA question-answer annotation files
* NExT-QA metadata resources
* Project configuration settings (config module)
* Shared utility modules and helper functions

---

## Outputs

All outputs are written locally during execution and exported to Google Drive at the end of the notebook as a complete experiment snapshot.

* Development-subset baseline prediction dataset
* Model-generated answers (Qwen2-VL-7B)
* Ground-truth answers
* Question and video metadata
* Inference timing metrics
* Baseline experiment summary report
* Representative prediction samples for qualitative inspection

---

## Processing Workflow

The notebook initializes the execution environment, loads configuration settings, clones the project repository, and restores the NExT-QA dataset if required. Dataset integrity and annotation coverage are validated before inference begins.

A development subset is sampled from the selected NExT-QA split to support rapid evaluation. Video files are resolved and paired with corresponding questions and ground-truth answers.

Baseline VideoQA inference is performed using Qwen2-VL-7B by passing sampled video frames directly to the model along with the associated question prompt. Model outputs, predictions, and runtime statistics are collected and stored in the local outputs directory.

After inference, summary statistics and evaluation reports are generated. Finally, all experiment outputs are exported as a structured snapshot to Google Drive for persistence and comparison with future representation-based pipelines.

---

## Notes

This notebook performs direct multimodal inference using Qwen2-VL-7B without any intermediate representation learning. Video inputs are processed as sampled frame sequences and passed directly into the model.

This baseline establishes the reference performance for comparison against CLIP-based and autoencoder-based representation learning pipelines. No learned embeddings or latent video representations are used in this stage.

All outputs are treated as part of a versioned experiment and are exported to Google Drive under a structured experiment directory.

### 🔷 Step 1 — Initialize Environment and Restore Dataset

* Initialize the notebook runtime and prepare the project execution environment.
* Clone the project repository using sparse checkout to minimize download size and startup overhead.
* Authenticate access to the private GitHub repository using a fine-grained access token stored in Google Colab Secrets.
* Load project configuration settings, utility modules, and required input/output paths.
* Mount Google Drive and restore the NExT-QA video dataset from the project release archive when needed.
* Verify local video cache availability and confirm the expected number of video files are present.
* Load NExT-QA question annotations and build the local video inventory.
* Validate annotation coverage and dataset readiness before VideoQA inference begins.
* Optionally display configuration details, dataset statistics, and validation summaries when `VERBOSE=True`.


In [ ]:
# ============================================================
# Step 1: Initialize Environment and Restore Dataset
# ============================================================

VERBOSE = True
REQUIRE_L4_GPU = True
EXPECTED_NEXTQA_VIDEO_COUNT = 5440

import os
import shutil
import time
from pathlib import Path

import pandas as pd

from google.colab import userdata, drive

print("Initializing notebook environment...")
print("-" * 60)

# ------------------------------------------------------------
# DRIVE MOUNT (ONCE ONLY)
# ------------------------------------------------------------

GOOGLE_DRIVE_MOUNT = "/content/drive"

if not os.path.exists(GOOGLE_DRIVE_MOUNT):
    print("Mounting Google Drive...")
    drive.mount(GOOGLE_DRIVE_MOUNT)
else:
    print("Google Drive already mounted.")

# ------------------------------------------------------------
# CLONE REPOSITORY (LOCAL EXECUTION ONLY)
# ------------------------------------------------------------

REPO_NAME = "videoqa-representation-comparison"
REPO_OWNER = "pgailinas"
REPO_BASE_DIR = "/content"
REPO_DIR = os.path.join(REPO_BASE_DIR, REPO_NAME)

github_token = userdata.get("GITHUB_TOKEN")

if github_token is None:
    raise ValueError("GITHUB_TOKEN not found in Colab Secrets.")

repo_url = (
    f"https://{github_token}"
    f"@github.com/{REPO_OWNER}/{REPO_NAME}.git"
)

os.chdir(REPO_BASE_DIR)

if not os.path.exists(REPO_DIR):

    print("Cloning project repository...")

    !git clone --quiet --filter=blob:none --no-checkout {repo_url}

    os.chdir(REPO_DIR)

    !git sparse-checkout init --cone
    !git sparse-checkout set src datasets outputs
    !git checkout --quiet main

else:

    print("Project repository already available.")
    os.chdir(REPO_DIR)

print(f"Repository ready: {REPO_DIR}")

# ------------------------------------------------------------
# LOAD CONFIG (SINGLE SOURCE OF TRUTH)
# ------------------------------------------------------------

print("\nLoading project configuration...")

from src.videoqa_representation_config import *

# IMPORTANT:
# All of the following MUST come from config file:
# - GOOGLE_DRIVE_ROOT
# - EXPERIMENTS_DRIVE_DIR
# - OUTPUTS_DIR
# - BASE_DIR

# ------------------------------------------------------------
# LOAD PROJECT MODULES
# ------------------------------------------------------------

from src.nextqa_video_cache import *
from src.nextqa_metadata import *
from src.video_segments import *
from src.training_validation import *
from src.training_metadata_io import *

required_paths = [
    Path("src"),
    Path("datasets"),
    Path("outputs"),
    QUESTIONS_DIR,
    METADATA_DIR,
]

missing_paths = [
    path for path in required_paths
    if not path.exists()
]

if missing_paths:
    for path in missing_paths:
        print(f"Missing required path: {path}")

    raise FileNotFoundError(
        "One or more required project paths are missing."
    )

for output_dir in [
    TRAINING_METADATA_DIR,
    TRAINING_REPORTS_DIR,
]:
    output_dir.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print("Project paths initialized.")

# ------------------------------------------------------------
# RESTORE VIDEO CACHE (FROM DRIVE ONLY IF NEEDED)
# ------------------------------------------------------------

print("\nChecking local NExT-QA video cache...")

existing_video_files = sorted(VIDEOS_DIR.rglob("*.mp4"))

if len(existing_video_files) == EXPECTED_NEXTQA_VIDEO_COUNT:

    print("Local video cache already available.")
    print(f"Videos found: {len(existing_video_files):,}")

else:

    print("Video cache missing — restoring from Google Drive...")

    # USE CONFIG PATHS ONLY (NO HARDCODED ROOTS)

    DRIVE_DATASET_DIR = GOOGLE_DRIVE_ROOT / "NExT-QA"
    DRIVE_RELEASES_DIR = DRIVE_DATASET_DIR / "releases"

    LOCAL_ARCHIVE_DIR = DATASET_DIR / "archives"
    LOCAL_ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

    COMBINED_ARCHIVE_NAME = "NExTVideo_combined.zip"

    DRIVE_COMBINED_ARCHIVE_PATH = DRIVE_RELEASES_DIR / COMBINED_ARCHIVE_NAME
    LOCAL_ARCHIVE_PATH = LOCAL_ARCHIVE_DIR / COMBINED_ARCHIVE_NAME

    if not DRIVE_COMBINED_ARCHIVE_PATH.exists():
        raise FileNotFoundError(
            f"Missing dataset archive in Drive: {DRIVE_COMBINED_ARCHIVE_PATH}"
        )

    print("Copying dataset archive from Drive...")

    shutil.copy2(
        DRIVE_COMBINED_ARCHIVE_PATH,
        LOCAL_ARCHIVE_PATH
    )

    print("Extracting video archive...")

    extract_nextqa_video_archive(
        combined_archive_path=LOCAL_ARCHIVE_PATH,
        local_videos_dir=VIDEOS_DIR,
        force_extract=False,
        verbose=VERBOSE,
    )

    print("Video cache restored.")

# ------------------------------------------------------------
# LOAD NEX T-QA METADATA
# ------------------------------------------------------------

print("\nLoading NExT-QA metadata and video inventory...")

split_annotations = load_nextqa_split_annotations(
    annotations_dir=QUESTIONS_DIR,
    verbose=VERBOSE,
)

annotations_df = combine_nextqa_annotations(
    split_dataframes=split_annotations,
    verbose=VERBOSE,
)

video_inventory_df = build_nextqa_video_inventory(
    videos_dir=VIDEOS_DIR,
    verbose=VERBOSE,
)

annotations_with_videos_df = attach_video_inventory_to_annotations(
    annotations=annotations_df,
    video_inventory=video_inventory_df,
    verbose=VERBOSE,
)

split_summary_df = summarize_nextqa_splits(
    annotations=annotations_df,
)

coverage_summary = verify_annotation_video_coverage(
    annotations=annotations_df,
    video_inventory=video_inventory_df,
    verbose=VERBOSE,
)

print("\nDataset metadata ready.")
print(f"Annotation records: {len(annotations_df):,}")
print(f"Video inventory   : {len(video_inventory_df):,}")

if VERBOSE:
    print("\nSplit Summary")
    print("-" * 60)
    display(split_summary_df)

print("\nEnvironment initialization complete.")
print("-" * 60)
print("Notebook is ready for baseline VideoQA inference.")



### 🔷 Step 2 — Define Development-Subset Inference Parameters

* Configure baseline VideoQA evaluation settings and experiment controls.
* Define the NExT-QA evaluation split used for development testing.
* Configure development-subset sampling parameters and randomization settings.
* Define the number of video frames sampled from each video during inference.
* Configure Qwen2-VL generation parameters, including output length and sampling behavior.
* Display the active baseline configuration used for the current experiment run.


In [ ]:
# ============================================================
# Step 2: Define Development-Subset Inference Parameters
# ============================================================

BASELINE_CONFIG = {
    # Evaluation control
    "evaluation_split": "val",
    "development_subset_size": 25,
    "random_seed": 42,

    # Development-subset experiments are used for
    # parameter optimization and workflow validation.

    # Answer mode
    # multiple_choice = Use NExT-QA answer choices
    # open_ended      = Generate free-form answers
    "answer_mode": "multiple_choice",

    # Video evidence selection
    "max_frames_per_question": 8,

    # Model generation settings
    "max_new_tokens": 64,
    "temperature": 0.0,
    "do_sample": False,

    # Output control
    "save_intermediate_results": True,
    "verbose": True,
}

print("\nBaseline Configuration:")
for key, value in BASELINE_CONFIG.items():
    print(f"  {key:<32}: {value}")



### 🔷 Step 3 — Verify GPU Runtime and Model Dependencies

* Verify that the Colab runtime satisfies Qwen2-VL-7B inference requirements.
* Confirm PyTorch installation and CUDA availability.
* Detect and display GPU hardware information, available memory, and runtime configuration.
* Verify that the selected GPU accelerator is suitable for baseline VideoQA inference.
* Confirm availability of required model, processor, and supporting software dependencies.
* Display environment validation results before loading the Qwen2-VL model.


In [ ]:
# ============================================================
# Step 3: Verify GPU Runtime and Model Dependencies
# ============================================================

import sys
import platform
import importlib

print("Verifying GPU runtime and model dependencies...\n")

# ------------------------------------------------------------
# Runtime Information
# ------------------------------------------------------------

print("Runtime Information")
print("-" * 60)
print(f"Python Version : {sys.version.split()[0]}")
print(f"Platform       : {platform.platform()}")

# ------------------------------------------------------------
# PyTorch / CUDA Verification
# ------------------------------------------------------------

try:
    import torch

    print("\nPyTorch Information")
    print("-" * 60)
    print(f"PyTorch Version : {torch.__version__}")
    print(f"CUDA Available  : {torch.cuda.is_available()}")

    if torch.cuda.is_available():

        print(f"CUDA Version    : {torch.version.cuda}")
        print(f"GPU Count       : {torch.cuda.device_count()}")

        for idx in range(torch.cuda.device_count()):

            gpu_name = torch.cuda.get_device_name(idx)

            gpu_props = torch.cuda.get_device_properties(idx)

            total_memory_gb = (
                gpu_props.total_memory /
                (1024 ** 3)
            )

            print(
                f"GPU {idx}          : "
                f"{gpu_name}"
            )

            print(
                f"GPU {idx} Memory   : "
                f"{total_memory_gb:.1f} GB"
            )

        allocated_gb = (
            torch.cuda.memory_allocated() /
            (1024 ** 3)
        )

        reserved_gb = (
            torch.cuda.memory_reserved() /
            (1024 ** 3)
        )

        print(
            f"Allocated Memory : "
            f"{allocated_gb:.2f} GB"
        )

        print(
            f"Reserved Memory  : "
            f"{reserved_gb:.2f} GB"
        )

        primary_gpu = torch.cuda.get_device_name(0)

        if REQUIRE_L4_GPU and "L4" not in primary_gpu:
            raise RuntimeError(
                f"Required NVIDIA L4 GPU not available. "
                f"Detected GPU: {primary_gpu}. "
                "Change the Colab runtime to L4 before continuing."
            )

        if "T4" in primary_gpu:

            print(
                "\nWARNING: NVIDIA T4 GPU detected "
                "(approximately 16 GB VRAM)."
            )

            print(
                "Large multimodal inference workloads "
                "may require reduced frame counts or "
                "memory optimization settings."
            )

        elif "L4" in primary_gpu:

            print(
                "\nNVIDIA L4 GPU detected "
                "(approximately 24 GB VRAM)."
            )

        device = "cuda"

    else:

        print("WARNING: No CUDA GPU detected.")
        device = "cpu"

except Exception as e:

    print(f"ERROR: Unable to load PyTorch ({e})")
    device = "cpu"

# ------------------------------------------------------------
# Required Packages
# ------------------------------------------------------------

required_packages = [
    "transformers",
    "accelerate",
    "torch",
    "torchvision",
    "numpy",
    "pandas",
    "PIL",
]

print("\nDependency Verification")
print("-" * 60)

dependency_status = []

for package_name in required_packages:

    try:
        module = importlib.import_module(package_name)

        version = getattr(module, "__version__", "unknown")

        dependency_status.append(
            {
                "package": package_name,
                "status": "OK",
                "version": version,
            }
        )

        print(f"[OK]   {package_name:<15} {version}")

    except Exception:
        dependency_status.append(
            {
                "package": package_name,
                "status": "MISSING",
                "version": "",
            }
        )

        print(f"[FAIL] {package_name}")

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

missing_packages = [
    item["package"]
    for item in dependency_status
    if item["status"] != "OK"
]

print("\nVerification Summary")
print("-" * 60)

print(f"Execution Device : {device}")

if len(missing_packages) == 0:
    print("All required dependencies are available.")
else:
    print("Missing packages:")
    for pkg in missing_packages:
        print(f"  - {pkg}")



### 🔷 Step 4 — Load Qwen2-VL-7B Model and Processor

* Load the Qwen2-VL-7B vision-language model used for baseline VideoQA inference.
* Load the associated processor for multimodal input preparation and prompt formatting.
* Configure model execution on the available GPU accelerator.
* Verify successful model and processor initialization.
* Display model loading status, device assignment, and memory utilization information.
* Confirm that the inference pipeline is ready for VideoQA evaluation.


In [ ]:
# ============================================================
# Step 4: Load Qwen2-VL-7B Model and Processor
# ============================================================

import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

print("Loading Qwen2-VL-7B model and processor...")

MODEL_ID = "Qwen/Qwen2-VL-7B-Instruct"

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is required for Qwen2-VL-7B inference. "
        "Please switch Colab runtime to GPU."
    )

device = "cuda"

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

model.eval()

print("Qwen2-VL-7B model and processor loaded successfully.")
print(f"Model ID : {MODEL_ID}")
print(f"Device   : {device}")
print(f"Dtype    : {model.dtype}")



### 🔷 Step 5 — Prepare Development Evaluation Dataset

* Select the configured NExT-QA evaluation split for baseline testing.
* Apply development-subset sampling limits to control experiment size and runtime.
* Validate required annotation fields, including video identifiers, questions, and answer labels.
* Resolve and verify local video file paths for all selected evaluation samples.
* Generate ground-truth answer text from the NExT-QA answer options.
* Validate evaluation dataset completeness and readiness for VideoQA inference.
* Prepare the development evaluation dataset used for baseline experimentation and parameter optimization.


In [ ]:
# ============================================================
# Step 5: Prepare Development Evaluation Dataset
# ============================================================

import random
from pathlib import Path

import pandas as pd

print("Preparing development evaluation subset...")

evaluation_split = BASELINE_CONFIG["evaluation_split"]
development_subset_size = BASELINE_CONFIG["development_subset_size"]
random_seed = BASELINE_CONFIG["random_seed"]

# ------------------------------------------------------------
# Select evaluation split
# ------------------------------------------------------------

required_annotation_columns = [
    "split",
    "video",
    "question",
    "answer",
    "a0",
    "a1",
    "a2",
    "a3",
    "a4",
]

missing_annotation_columns = [
    col for col in required_annotation_columns
    if col not in annotations_df.columns
]

if missing_annotation_columns:
    raise ValueError(
        f"annotations_df is missing required columns: {missing_annotation_columns}"
    )

eval_df = annotations_df[
    annotations_df["split"] == evaluation_split
].copy()

if len(eval_df) == 0:
    raise ValueError(f"No records found for split: {evaluation_split}")

print(f"Development subset size: {development_subset_size:,}")

sample_size = min(
    development_subset_size,
    len(eval_df),
)

eval_df = (
    eval_df
    .sample(
        n=sample_size,
        random_state=random_seed,
    )
    .reset_index(drop=True)
)

print(f"Selected evaluation samples: {len(eval_df):,}")

# ------------------------------------------------------------
# Attach video file paths
# ------------------------------------------------------------

VIDEO_DIR = (
    Path(REPO_DIR)
    / "datasets"
    / "NExT-QA"
    / "videos"
)

def resolve_video_path(video_id):
    matches = list(VIDEO_DIR.rglob(f"{video_id}.mp4"))

    if len(matches) == 0:
        return None

    return matches[0]

eval_df["video_path"] = eval_df["video"].apply(resolve_video_path)

missing_video_count = eval_df["video_path"].isna().sum()

print(f"Missing video files: {missing_video_count}")

if missing_video_count > 0:
    display(eval_df[eval_df["video_path"].isna()].head())

    raise FileNotFoundError(
        "One or more evaluation samples do not have matching video files."
    )

# ------------------------------------------------------------
# Attach ground-truth answer text
# ------------------------------------------------------------

def answer_index_to_text(row):
    answer_idx = int(row["answer"])
    option_col = f"a{answer_idx}"

    if option_col not in row.index:
        raise ValueError(
            f"Answer option column not found: {option_col}"
        )

    return row[option_col]

eval_df["ground_truth_text"] = (
    eval_df.apply(
        answer_index_to_text,
        axis=1,
    )
)

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

required_eval_columns = [
    "video",
    "question",
    "answer",
    "a0",
    "a1",
    "a2",
    "a3",
    "a4",
    "ground_truth_text",
    "video_path",
]

missing_columns = [
    col for col in required_eval_columns
    if col not in eval_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required evaluation columns: {missing_columns}"
    )

print("\nEvaluation dataset prepared successfully.")
print(f"Evaluation samples : {len(eval_df):,}")
print(f"Evaluation split   : {evaluation_split}")
print(f"Random seed        : {random_seed}")
print(f"Answer mode        : {BASELINE_CONFIG['answer_mode']}")

print("\nEvaluation Dataset Preview:")
display(eval_df.head())



### 🔷 Step 6 — Run Development-Subset Baseline VideoQA Inference

* Sample representative video frames from each evaluation video.
* Construct multimodal prompts consisting of sampled video frames and associated questions.
* Execute baseline VideoQA inference using Qwen2-VL-7B on the development evaluation dataset.
* Generate predicted answers for each evaluation sample.
* Record inference results, runtime statistics, and processing outcomes.
* Monitor and manage GPU memory utilization throughout inference execution.


In [ ]:
# ============================================================
# Step 6: Run Development-Subset Baseline VideoQA Inference
# ============================================================

import time
import gc
import re
import pandas as pd
from tqdm.notebook import tqdm
from PIL import Image
import cv2
import torch

# ------------------------------------------------------------
# Helper Functions
# ------------------------------------------------------------

def clear_gpu_memory():
    """
    Release unused Python and CUDA memory between inference samples.
    """

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def sample_video_frames(
    video_path,
    num_frames=8
):
    """
    Uniformly sample frames from a video.
    """

    cap = cv2.VideoCapture(str(video_path))

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if frame_count <= 0:
        cap.release()
        return []

    frame_indices = [
        int(i * frame_count / num_frames)
        for i in range(num_frames)
    ]

    frames = []

    for idx in frame_indices:

        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)

        success, frame = cap.read()

        if success:
            frame = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )

            frames.append(
                Image.fromarray(frame)
            )

    cap.release()

    return frames


def build_videoqa_prompt(
    row,
    answer_mode
):
    """
    Build the text prompt for either multiple-choice or open-ended VideoQA.
    """

    question = row["question"]

    if answer_mode == "multiple_choice":

        choice_text = (
            f"0. {row['a0']}\n"
            f"1. {row['a1']}\n"
            f"2. {row['a2']}\n"
            f"3. {row['a3']}\n"
            f"4. {row['a4']}"
        )

        return (
            "Answer the video question by selecting the best answer choice.\n"
            "Respond with only the number of the best answer choice.\n\n"
            f"Question: {question}\n\n"
            f"Choices:\n{choice_text}"
        )

    if answer_mode == "open_ended":

        return (
            "Answer the following video question "
            "as concisely as possible.\n\n"
            f"Question: {question}"
        )

    raise ValueError(
        f"Unsupported answer_mode: {answer_mode}"
    )


def extract_predicted_choice(
    prediction_text
):
    """
    Extract a predicted multiple-choice answer index from model output.

    Expected choices are 0, 1, 2, 3, or 4.
    """

    if prediction_text is None:
        return None

    text = str(prediction_text).strip()

    # Prefer a standalone digit at the beginning of the response.
    leading_match = re.match(r"^\s*([0-4])\b", text)

    if leading_match:
        return int(leading_match.group(1))

    # Fallback: any standalone valid choice digit.
    any_match = re.search(r"\b([0-4])\b", text)

    if any_match:
        return int(any_match.group(1))

    return None


def run_videoqa_inference(
    video_path,
    row
):
    """
    Execute baseline Qwen2-VL inference.
    """

    frames = None
    messages = None
    text = None
    inputs = None
    generated_ids = None
    generated_ids_trimmed = None
    output_text = None

    try:

        frames = sample_video_frames(
            video_path,
            BASELINE_CONFIG["max_frames_per_question"]
        )

        if len(frames) == 0:
            return "VIDEO_READ_ERROR"

        prompt_text = build_videoqa_prompt(
            row=row,
            answer_mode=BASELINE_CONFIG["answer_mode"],
        )

        messages = [
            {
                "role": "user",
                "content": (
                    [{"type": "image", "image": frame}
                     for frame in frames]
                    +
                    [{
                        "type": "text",
                        "text": prompt_text,
                    }]
                )
            }
        ]

        text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = processor(
            text=[text],
            images=frames,
            return_tensors="pt"
        )

        inputs = {
            k: v.to(model.device)
            for k, v in inputs.items()
        }

        generate_kwargs = {
            "max_new_tokens": BASELINE_CONFIG["max_new_tokens"],
            "do_sample": BASELINE_CONFIG["do_sample"],
        }

        if BASELINE_CONFIG["do_sample"]:
            generate_kwargs["temperature"] = BASELINE_CONFIG["temperature"]

        with torch.no_grad():

            generated_ids = model.generate(
                **inputs,
                **generate_kwargs
            )

        generated_ids_trimmed = [
            output_ids[len(input_ids):]
            for input_ids, output_ids in zip(
                inputs["input_ids"],
                generated_ids
            )
        ]

        output_text = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )[0]

        return output_text.strip()

    finally:

        del frames
        del messages
        del text
        del inputs
        del generated_ids
        del generated_ids_trimmed
        del output_text

        clear_gpu_memory()


# ------------------------------------------------------------
# Baseline Inference Loop
# ------------------------------------------------------------

answer_mode = BASELINE_CONFIG["answer_mode"]

print(
    f"Running development-subset baseline inference "
    f"on {len(eval_df):,} samples..."
)
print(f"Answer mode: {answer_mode}")

results = []

clear_gpu_memory()

start_time = time.time()

for _, row in tqdm(
    eval_df.iterrows(),
    total=len(eval_df),
    desc="Running Baseline VideoQA"
):

    try:

        prediction = run_videoqa_inference(
            video_path=row["video_path"],
            row=row,
        )

    except Exception as e:

        prediction = f"ERROR: {str(e)}"
        clear_gpu_memory()

    result = {
        "video": row["video"],
        "question": row["question"],
        "ground_truth": row["ground_truth_text"],
        "prediction": prediction,
        "answer_mode": answer_mode,
    }

    if answer_mode == "multiple_choice":

        ground_truth_choice = int(row["answer"])
        predicted_choice = extract_predicted_choice(
            prediction
        )

        result.update({
            "ground_truth_choice": ground_truth_choice,
            "predicted_choice": predicted_choice,
            "choice_correct": (
                predicted_choice == ground_truth_choice
                if predicted_choice is not None
                else False
            ),
        })

    results.append(result)

elapsed_time = time.time() - start_time

prediction_df = pd.DataFrame(results)

print(f"Evaluation samples : {len(prediction_df):,}")
print(f"Elapsed time       : {elapsed_time:.1f} seconds")
print(
    f"Average/sample     : "
    f"{elapsed_time / len(prediction_df):.2f} seconds"
)

if answer_mode == "multiple_choice":
    valid_choice_count = prediction_df["predicted_choice"].notna().sum()
    correct_choice_count = prediction_df["choice_correct"].sum()
    choice_accuracy = correct_choice_count / len(prediction_df)

    print("\nMultiple-Choice Results")
    print("-" * 60)
    print(f"Valid choice predictions : {valid_choice_count:,}")
    print(f"Correct choice predictions: {correct_choice_count:,}")
    print(f"Choice accuracy          : {choice_accuracy:.2%}")

display(prediction_df.head())



### 🔷 Step 7 — Validate Prediction Results

* Verify that baseline prediction records were generated successfully.
* Validate required prediction fields and output structure.
* Check for missing, empty, or invalid prediction values.
* Identify inference failures, video processing errors, and runtime exceptions.
* Generate prediction validation statistics and summary metrics.
* Confirm prediction dataset integrity before saving experiment results.


In [ ]:
# ============================================================
# Step 7: Validate Prediction Results
# ============================================================

import pandas as pd

print("Validating baseline prediction results...")

if "prediction_df" not in globals():
    raise NameError("prediction_df was not found. Run Step 6 first.")

answer_mode = BASELINE_CONFIG["answer_mode"]

required_prediction_columns = [
    "video",
    "question",
    "ground_truth",
    "prediction",
    "answer_mode",
]

if answer_mode == "multiple_choice":
    required_prediction_columns.extend(
        [
            "ground_truth_choice",
            "predicted_choice",
            "choice_correct",
        ]
    )

missing_columns = [
    col for col in required_prediction_columns
    if col not in prediction_df.columns
]

if missing_columns:
    raise ValueError(f"Missing required prediction columns: {missing_columns}")

validation_summary = {
    "total_predictions": len(prediction_df),
    "missing_predictions": prediction_df["prediction"].isna().sum(),
    "empty_predictions": (
        prediction_df["prediction"].astype(str).str.strip() == ""
    ).sum(),
    "error_predictions": (
        prediction_df["prediction"].astype(str).str.startswith("ERROR")
    ).sum(),
    "video_read_errors": (
        prediction_df["prediction"].astype(str) == "VIDEO_READ_ERROR"
    ).sum(),
    "unique_videos": prediction_df["video"].nunique(),
}

if answer_mode == "multiple_choice":
    validation_summary.update(
        {
            "valid_choice_predictions": prediction_df[
                "predicted_choice"
            ].notna().sum(),
            "invalid_choice_predictions": prediction_df[
                "predicted_choice"
            ].isna().sum(),
            "correct_choice_predictions": prediction_df[
                "choice_correct"
            ].sum(),
            "choice_accuracy": prediction_df[
                "choice_correct"
            ].mean(),
        }
    )

validation_df = pd.DataFrame(
    validation_summary.items(),
    columns=["Validation Check", "Count"],
)

display(validation_df)

problem_mask = (
    prediction_df["prediction"].isna()
    | (prediction_df["prediction"].astype(str).str.strip() == "")
    | (prediction_df["prediction"].astype(str).str.startswith("ERROR"))
    | (prediction_df["prediction"].astype(str) == "VIDEO_READ_ERROR")
)

if answer_mode == "multiple_choice":
    problem_mask = (
        problem_mask
        | prediction_df["predicted_choice"].isna()
    )

problem_predictions_df = prediction_df[
    problem_mask
].copy()

if len(problem_predictions_df) > 0:
    print("\nProblem predictions detected:")
    display(problem_predictions_df)
else:
    print(
        "\nPrediction validation passed. "
        "No missing, empty, error, or invalid choice predictions detected."
    )

# ------------------------------------------------------------
# Runtime Projection
# ------------------------------------------------------------

if "elapsed_time" in globals():
    avg_time_per_sample = elapsed_time / len(prediction_df)
    total_dataset_size = len(annotations_df)
    projected_seconds = avg_time_per_sample * total_dataset_size
    projected_hours = projected_seconds / 3600

    print("\nRuntime Projection")
    print("-" * 60)
    print(f"Average Time per Sample : {avg_time_per_sample:.2f} sec")
    print(f"Dataset Size            : {total_dataset_size:,}")
    print(f"Projected Runtime       : {projected_hours:.2f} hours")



### 🔷 Step 8 — Save Prediction Results

* Save baseline VideoQA prediction records to persistent output storage.
* Create required output directories when necessary.
* Verify successful file creation and storage operations.
* Record prediction output file locations for subsequent analysis and evaluation.
* Confirm that saved prediction results are available for downstream reporting workflows.


In [ ]:
# ============================================================
# Step 8 Save Prediction Results
# ============================================================

from pathlib import Path

if "prediction_df" not in globals():
    raise NameError("prediction_df was not found. Run Step 11 first.")

BASELINE_OUTPUT_DIR = (
    Path(REPO_DIR)
    / "outputs"
    / "baseline"
)
BASELINE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

prediction_output_file = BASELINE_OUTPUT_DIR / "baseline_predictions.csv"

prediction_df.to_csv(
    prediction_output_file,
    index=False
)

print("Baseline prediction results saved successfully.")
print(f"Prediction file : {prediction_output_file}")
print(f"Records saved   : {len(prediction_df):,}")



### 🔷 Step 9 — Generate Development-Subset Baseline Summary Report

* Compute summary statistics describing baseline VideoQA execution results.
* Aggregate prediction counts, validation metrics, and runtime statistics.
* Estimate execution requirements for larger evaluation datasets and full-experiment runs.
* Generate experiment summary information for development-subset analysis.
* Save baseline summary reports for later comparison with autoencoder-based VideoQA experiments.


In [ ]:
# ============================================================
# Step 9: Generate Development-Subset Baseline Summary Report
# ============================================================

import pandas as pd
from pathlib import Path

if "prediction_df" not in globals():
    raise NameError("prediction_df was not found. Run Step 6 first.")

if "annotations_df" not in globals():
    raise NameError("annotations_df was not found.")

if "elapsed_time" not in globals():
    raise NameError("elapsed_time was not found.")

answer_mode = BASELINE_CONFIG["answer_mode"]

BASELINE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

summary_output_file = BASELINE_SUMMARY_CSV

# ------------------------------------------------------------
# Prediction Statistics
# ------------------------------------------------------------

total_predictions = len(prediction_df)

missing_predictions = (
    prediction_df["prediction"]
    .isna()
    .sum()
)

empty_predictions = (
    prediction_df["prediction"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

error_predictions = (
    prediction_df["prediction"]
    .astype(str)
    .str.startswith("ERROR")
    .sum()
)

video_read_errors = (
    prediction_df["prediction"]
    .astype(str)
    .eq("VIDEO_READ_ERROR")
    .sum()
)

valid_predictions = (
    total_predictions
    - missing_predictions
    - empty_predictions
    - error_predictions
    - video_read_errors
)

summary_rows = [
    {
        "metric": "answer_mode",
        "value": answer_mode,
    },
    {
        "metric": "total_predictions",
        "value": total_predictions,
    },
    {
        "metric": "valid_predictions",
        "value": valid_predictions,
    },
    {
        "metric": "missing_predictions",
        "value": missing_predictions,
    },
    {
        "metric": "empty_predictions",
        "value": empty_predictions,
    },
    {
        "metric": "error_predictions",
        "value": error_predictions,
    },
    {
        "metric": "video_read_errors",
        "value": video_read_errors,
    },
    {
        "metric": "unique_videos",
        "value": prediction_df["video"].nunique(),
    },
]

# ------------------------------------------------------------
# Multiple-Choice Statistics
# ------------------------------------------------------------

if answer_mode == "multiple_choice":

    valid_choice_predictions = (
        prediction_df["predicted_choice"]
        .notna()
        .sum()
    )

    invalid_choice_predictions = (
        prediction_df["predicted_choice"]
        .isna()
        .sum()
    )

    correct_choice_predictions = (
        prediction_df["choice_correct"]
        .sum()
    )

    choice_accuracy = (
        correct_choice_predictions / total_predictions
        if total_predictions > 0
        else 0
    )

    summary_rows.extend(
        [
            {
                "metric": "valid_choice_predictions",
                "value": valid_choice_predictions,
            },
            {
                "metric": "invalid_choice_predictions",
                "value": invalid_choice_predictions,
            },
            {
                "metric": "correct_choice_predictions",
                "value": correct_choice_predictions,
            },
            {
                "metric": "choice_accuracy",
                "value": round(choice_accuracy, 4),
            },
        ]
    )

# ------------------------------------------------------------
# Runtime Statistics
# ------------------------------------------------------------

avg_time_per_sample = (
    elapsed_time / total_predictions
)

total_dataset_size = len(annotations_df)

projected_full_dataset_seconds = (
    avg_time_per_sample
    * total_dataset_size
)

projected_full_dataset_hours = (
    projected_full_dataset_seconds
    / 3600
)

evaluation_split_size = len(
    annotations_df[
        annotations_df["split"]
        == BASELINE_CONFIG["evaluation_split"]
    ]
)

projected_eval_split_minutes = (
    (avg_time_per_sample * evaluation_split_size)
    / 60
)

summary_rows.extend(
    [
        {
            "metric": "elapsed_time_seconds",
            "value": round(elapsed_time, 2),
        },
        {
            "metric": "average_time_per_sample_seconds",
            "value": round(avg_time_per_sample, 2),
        },
        {
            "metric": "projected_validation_runtime_minutes",
            "value": round(
                projected_eval_split_minutes,
                2,
            ),
        },
        {
            "metric": "projected_full_dataset_runtime_hours",
            "value": round(
                projected_full_dataset_hours,
                2,
            ),
        },
    ]
)

# ------------------------------------------------------------
# Summary Report
# ------------------------------------------------------------

baseline_summary_df = pd.DataFrame(summary_rows)

baseline_summary_df.to_csv(
    summary_output_file,
    index=False,
)

print(
    f"Baseline summary report saved: "
    f"{summary_output_file}"
)

display(baseline_summary_df)



### 🔷 Step 10 — Display Sample Predictions

* Randomly select representative prediction records from the baseline evaluation results.
* Display evaluation questions, ground-truth answers, and model predictions.
* Review prediction quality and response characteristics across selected samples.
* Support qualitative assessment of baseline VideoQA performance.
* Provide example results for experiment verification and debugging purposes.


In [ ]:
# ============================================================
# Step 10: Display Sample Predictions
# ============================================================

import pandas as pd

if "prediction_df" not in globals():
    raise NameError("prediction_df was not found. Run Step 6 first.")

sample_count = min(
    10,
    len(prediction_df),
)

sample_predictions_df = (
    prediction_df
    .sample(
        n=sample_count,
        random_state=BASELINE_CONFIG["random_seed"],
    )
    .reset_index(drop=True)
)

answer_mode = BASELINE_CONFIG["answer_mode"]

display_columns = [
    "video",
    "question",
    "ground_truth",
    "prediction",
]

if answer_mode == "multiple_choice":

    display_columns.extend(
        [
            "ground_truth_choice",
            "predicted_choice",
            "choice_correct",
        ]
    )

print(f"Displaying {sample_count} sample predictions...")
print(f"Answer mode: {answer_mode}\n")

display(
    sample_predictions_df[
        display_columns
    ]
)

# ------------------------------------------------------------
# Multiple-Choice Summary
# ------------------------------------------------------------

if answer_mode == "multiple_choice":

    valid_choice_predictions = (
        prediction_df["predicted_choice"]
        .notna()
        .sum()
    )

    correct_choice_predictions = (
        prediction_df["choice_correct"]
        .sum()
    )

    choice_accuracy = (
        correct_choice_predictions
        / len(prediction_df)
    )

    print("\nMultiple-Choice Summary")
    print("-" * 60)
    print(
        f"Valid Choice Predictions : "
        f"{valid_choice_predictions:,}"
    )
    print(
        f"Correct Predictions      : "
        f"{correct_choice_predictions:,}"
    )
    print(
        f"Choice Accuracy          : "
        f"{choice_accuracy:.2%}"
    )



### 🔷 Step 11 — Export Baseline Outputs to Google Drive

* Copy locally generated baseline evaluation outputs from the Colab runtime to Google Drive.
* Export prediction results and summary evaluation reports as persistent experiment artifacts.
* Store baseline outputs under the experiment-specific directory in Google Drive for reproducibility.


In [ ]:
# ============================================================
# Step 11: Export Baseline Outputs to Google Drive
# ============================================================

import shutil
from pathlib import Path

print("Exporting full experiment directory to Google Drive...")
print("-" * 60)

# ------------------------------------------------------------
# SOURCE (local experiment outputs)
# ------------------------------------------------------------

local_experiment_outputs = Path(
    "/content/videoqa-representation-comparison/outputs"
)

# ------------------------------------------------------------
# DESTINATION (Drive experiment folder)
# ------------------------------------------------------------
EXPERIMENT_NAME = "Run_Qwen2VL_Baseline"

drive_experiment_outputs = (
    EXPERIMENTS_DRIVE_DIR / EXPERIMENT_NAME / "outputs"
)

drive_experiment_outputs.parent.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# COPY ENTIRE OUTPUT DIRECTORY (SNAPSHOT STYLE)
# ------------------------------------------------------------

shutil.copytree(
    local_experiment_outputs,
    drive_experiment_outputs,
    dirs_exist_ok=True
)

print("Export complete.")
print(f"Local: {local_experiment_outputs}")
print(f"Drive: {drive_experiment_outputs}")

